# PDF OCR to Knowledge Base - v3 (Production Ready)

**Version History:**
- v1: Initial implementation with response handling issues
- v2: Test notebook with fixes
- v3: Production-ready with all improvements integrated

**Key Fixes in v3:**
✅ Robust response.content handling (string or list)
✅ Proper error handling throughout pipeline
✅ Rate limiting between API calls
✅ Document hierarchy preservation
✅ Full batch processing with progress tracking
✅ Output validation and statistics

**Based on:**
- LangChain Google v1.33+
- Google GenAI SDK latest
- Gemini 3-Flash-Preview model

In [4]:
import fitz  # PyMuPDF
import os
import base64
from PIL import Image
import io
from dotenv import load_dotenv
import time
from datetime import datetime
from typing import List, Tuple

# LangChain imports
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

# --- CONFIGURATION ---
# Load environment variables (for API key)
load_dotenv()

# Get API key with validation
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables. Please set it in your .env file.")

# PDF Configuration
pdf_file = "CourseBook_Semester3_AlTafsir.pdf"        # Path to your input PDF
output_file = "output_ocr_CourseBook_Semester3_AlTafsir.md"  # Output file (Markdown format)
start_page = 1                # Page number to start processing (1-based)
end_page = 102               # Page number to end processing (1-based)

# API Configuration
MODEL = "gemini-3-flash-preview"
TEMPERATURE = 0.3
RATE_LIMIT_DELAY = 1.5  # seconds between API calls

print("="*60)
print("PDF OCR to Knowledge Base - v3")
print("="*60)
print(f"✓ Imports successful")
print(f"✓ API Key: {GOOGLE_API_KEY[:10]}...")
print(f"✓ Model: {MODEL}")
print(f"✓ Config: Pages {start_page}-{end_page}")
print("="*60)

PDF OCR to Knowledge Base - v3
✓ Imports successful
✓ API Key: AIzaSyBEDl...
✓ Model: gemini-3-flash-preview
✓ Config: Pages 1-102


## Helper Functions

In [5]:
def image_to_base64(image: Image.Image, format="JPEG") -> str:
    """
    Converts PIL Image to base64 string.
    Handles different image modes (RGBA, Palette, RGB).
    """
    try:
        # Convert image modes to RGB for consistency
        if image.mode == 'RGBA':
            bg = Image.new('RGB', image.size, (255, 255, 255))
            bg.paste(image, (0, 0), image)
            image = bg
        elif image.mode == 'P':  # Palette mode
            image = image.convert('RGB')
        elif image.mode not in ['RGB', 'L']:
            image = image.convert('RGB')

        buffered = io.BytesIO()
        image.save(buffered, format=format)
        img_bytes = buffered.getvalue()
        return base64.b64encode(img_bytes).decode('utf-8')
    except Exception as e:
        raise ValueError(f"Failed to convert image to base64: {e}")


def extract_text_from_response(response) -> str:
    """
    Extracts text from Gemini API response.
    CRITICAL FIX: Handles both string and list responses.
    
    Response formats from LangChain ChatGoogleGenerativeAI:
    - String: "extracted text"
    - List: [{"text": "part1"}, {"text": "part2"}] or ["part1", "part2"]
    - Mixed: ["part1", {"text": "part2"}]
    """
    if not hasattr(response, 'content'):
        raise ValueError("Response has no 'content' attribute")
    
    content = response.content
    
    # Case 1: String response (most common)
    if isinstance(content, str):
        return content
    
    # Case 2: List response (multimodal or model variations)
    if isinstance(content, list):
        text_parts = []
        for item in content:
            if isinstance(item, str):
                text_parts.append(item)
            elif isinstance(item, dict):
                if 'text' in item:
                    text_parts.append(item['text'])
        
        if not text_parts:
            raise ValueError("List response contains no text items")
        
        return ''.join(text_parts)
    
    # Case 3: Fallback - convert to string
    return str(content)

## OCR Core Function

In [6]:
def get_ocr_text_from_image(image_base64: str, api_key: str) -> Tuple[bool, str]:
    """
    Sends image to Gemini for OCR with robust error handling.
    
    Returns:
        Tuple[success: bool, text: str]
        - (True, extracted_text) on success
        - (False, error_message) on failure
    """
    try:
        # Initialize LLM
        llm = ChatGoogleGenerativeAI(
            model=MODEL,
            google_api_key=api_key,
            temperature=TEMPERATURE
        )
        
        # Create multimodal message
        message = HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": (
                        "Perform detailed OCR on this image while carefully preserving the document structure. "
                        "1. Identify text hierarchy (main titles, subheadings, body text, quotes) "
                        "2. Preserve ALL formatting (lists, bullet points, numbering, indentation) "
                        "3. Maintain text colors and special formatting (colored text, highlighted sections) "
                        "4. Format output in markdown with appropriate heading levels (# for main titles, ## for subtitles, etc.) "
                        "5. Extract ALL text exactly as written without summarization "
                        "6. For religious/colored text, preserve color indications in markdown (e.g., green text in brackets) "
                        "If there is no text in the image, respond with exactly: [No text found in image]"
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{image_base64}",
                },
            ]
        )
        
        # Rate limiting
        time.sleep(RATE_LIMIT_DELAY)
        
        # Get response with timeout handling
        try:
            response = llm.invoke([message])
        except Exception as e:
            if "rate limit" in str(e).lower():
                return (False, f"[Rate Limited - {str(e)}]")
            raise
        
        # Extract text robustly
        try:
            text = extract_text_from_response(response)
        except Exception as e:
            return (False, f"[Response Parsing Failed: {e}]")
        
        # Check for empty result
        if not text or text.strip() == "":
            return (False, "[Empty response from API]")
        
        # Check for "no text" marker
        if "[no text found in image]" in text.lower():
            return (False, "[No text found in image]")
        
        return (True, text.strip())
        
    except Exception as e:
        error_msg = f"[OCR Error: {type(e).__name__}: {str(e)[:100]}]"
        return (False, error_msg)

## PDF Processing Pipeline

In [7]:
def process_pdf_page_with_ocr(page: fitz.Page, llm_api_key: str) -> Tuple[int, int, str]:
    """
    Process all images on a PDF page with OCR.
    
    Returns:
        Tuple[successful_count, failed_count, combined_text]
    """
    successful = 0
    failed = 0
    page_text = ""
    
    # Get all images on page
    image_list = page.get_images(full=True)
    
    if not image_list:
        return (0, 0, "")
    
    for img_index, img_info in enumerate(image_list, 1):
        xref = img_info[0]
        
        try:
            # Extract image from PDF
            base_image = page.parent.extract_image(xref)
            if not base_image:
                failed += 1
                continue
            
            image_bytes = base_image.get("image")
            if not image_bytes:
                failed += 1
                continue
            
            # Convert to PIL and encode
            pil_image = Image.open(io.BytesIO(image_bytes))
            img_base64 = image_to_base64(pil_image, format="JPEG")
            
            # Perform OCR
            success, ocr_text = get_ocr_text_from_image(img_base64, llm_api_key)
            
            if success:
                page_text += ocr_text + "\n\n"
                successful += 1
            else:
                failed += 1
                
        except Exception as e:
            failed += 1
    
    return (successful, failed, page_text)

## Main Orchestration Function

In [8]:
def pdf_to_ocr_knowledge_base(pdf_path: str, output_path: str, start_pg: int, end_pg: int, api_key: str):
    """
    Main workflow: Extract text from PDF using OCR and save to markdown.
    
    Features:
    - Input validation
    - Page range handling
    - Progress tracking
    - Statistics reporting
    - Error resilience
    """
    # Validate inputs
    if not os.path.exists(pdf_path):
        print(f"ERROR: PDF file not found: {pdf_path}")
        return False
    
    if not api_key:
        print("ERROR: API key is missing")
        return False
    
    print(f"\n{'='*60}")
    print(f"Starting PDF Processing")
    print(f"{'='*60}")
    print(f"PDF: {pdf_path}")
    print(f"Output: {output_path}")
    
    # Open PDF and validate page range
    try:
        doc = fitz.open(pdf_path)
        total_pages = len(doc)
        print(f"Total pages in PDF: {total_pages}")
    except Exception as e:
        print(f"ERROR: Failed to open PDF: {e}")
        return False
    
    # Normalize page range
    actual_start = max(1, min(start_pg, total_pages))
    actual_end = min(max(start_pg, end_pg), total_pages)
    
    if actual_start > actual_end:
        print(f"ERROR: Invalid page range: {start_pg}-{end_pg}")
        doc.close()
        return False
    
    print(f"Processing pages: {actual_start}-{actual_end}")
    
    # Initialize statistics
    stats = {
        "pages_processed": 0,
        "pages_successful": 0,
        "pages_failed": 0,
        "images_successful": 0,
        "images_failed": 0,
        "total_characters": 0,
    }
    
    # Build output with header
    final_output = f"# Knowledge Base - {os.path.basename(pdf_path)}\n\n"
    final_output += f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    final_output += f"**Pages Processed:** {actual_start}-{actual_end}\n"
    final_output += f"**Model:** {MODEL}\n\n"
    final_output += "---\n\n"
    
    # Process each page
    start_time = time.time()
    
    for page_num in range(actual_start, actual_end + 1):
        try:
            page = doc.load_page(page_num - 1)
            
            # Process page
            success_count, fail_count, page_text = process_pdf_page_with_ocr(page, api_key)
            
            # Update statistics
            stats["pages_processed"] += 1
            if success_count > 0 or fail_count == 0:
                stats["pages_successful"] += 1
            else:
                stats["pages_failed"] += 1
            
            stats["images_successful"] += success_count
            stats["images_failed"] += fail_count
            
            # Add to output if content found
            if page_text.strip():
                final_output += f"## Page {page_num}\n\n{page_text}\n\n"
                stats["total_characters"] += len(page_text)
            
            # Progress update
            elapsed = time.time() - start_time
            pages_done = page_num - actual_start + 1
            pages_total = actual_end - actual_start + 1
            rate = pages_done / elapsed if elapsed > 0 else 0
            eta_secs = (pages_total - pages_done) / rate if rate > 0 else 0
            
            print(f"[{pages_done}/{pages_total}] Page {page_num}: {success_count} OK, {fail_count} FAIL | "
                  f"ETA: {int(eta_secs//60)}m {int(eta_secs%60)}s")
            
        except Exception as e:
            print(f"[ERROR] Page {page_num}: {e}")
            stats["pages_failed"] += 1
    
    doc.close()
    
    # Add statistics to output
    final_output += f"\n---\n\n## Processing Statistics\n\n"
    final_output += f"- **Pages Processed:** {stats['pages_processed']}\n"
    final_output += f"- **Pages Successful:** {stats['pages_successful']}\n"
    final_output += f"- **Pages Failed:** {stats['pages_failed']}\n"
    final_output += f"- **Images Successfully Extracted:** {stats['images_successful']}\n"
    final_output += f"- **Images Failed:** {stats['images_failed']}\n"
    final_output += f"- **Total Characters Extracted:** {stats['total_characters']:,}\n"
    
    # Save output
    try:
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(final_output)
        print(f"\n✓ Successfully saved to: {output_path}")
        print(f"✓ Total size: {len(final_output):,} characters")
    except Exception as e:
        print(f"ERROR: Failed to save output: {e}")
        return False
    
    elapsed = time.time() - start_time
    print(f"✓ Processing completed in {int(elapsed//60)}m {int(elapsed%60)}s")
    print(f"{'='*60}\n")
    
    return True

## Execute Full PDF Processing

In [9]:
# # PDF Configuration
# pdf_file = "CourseBook_Semester3_AlTafsir.pdf"        # Path to your input PDF
# output_file = "output_ocr_CourseBook_Semester3_AlTafsir.md"  # Output file (Markdown format)
# start_page = 1                # Page number to start processing (1-based)
# end_page = 4               # Page number to end processing (1-based)
# # Execute the full workflow
# if __name__ == "__main__":
#     # Validate API key
#     if not GOOGLE_API_KEY:
#         print("ERROR: GOOGLE_API_KEY not set. Cannot proceed.")
#     else:
#         # Run the PDF processing
#         success = pdf_to_ocr_knowledge_base(
#             pdf_file, 
#             output_file, 
#             start_page, 
#             end_page, 
#             GOOGLE_API_KEY
#         )
        
#         if success:
#             print("✓ PDF processing completed successfully!")
#         else:
#             print("✗ PDF processing failed. Check errors above.")

## Diagnostic & Analysis

### Expected Behavior Analysis

Based on test runs and API documentation, here's what should happen:

**Test Notebook Findings:**
- ✅ PDF loading works correctly
- ✅ Image extraction from PDF works  
- ✅ Base64 encoding works
- ⚠️ API response handling had issues with list vs string responses
- ✅ `extract_text_from_response()` function handles both cases

**Key Insight from Error:** `'list' object has no attribute 'lower'`
- This occurred when trying to call `.lower()` directly on response.content
- **Solution:** Always extract text safely first, then perform string operations

**V3 Improvements:**
1. Returns `Tuple[bool, str]` instead of just string - clearer success/failure
2. Rate limiting properly implemented
3. Progress tracking with ETA
4. Statistics collection
5. Comprehensive error messages
6. No direct `.lower()` calls on response.content

In [10]:
# Show the critical difference between V2 (broken) and V3 (fixed)

comparison = """
╔════════════════════════════════════════════════════════════════════════════╗
║                     CRITICAL FIX: Response Handling                        ║
╚════════════════════════════════════════════════════════════════════════════╝

❌ V2 (BROKEN):
──────────────────────────────────────────────────────────────────────────────
response = llm.invoke([message])
if '[no text found in image]' in response.content.lower():  # ← CRASH HERE
    return "[No text found in image]"
return response.content

ERROR: 'list' object has no attribute 'lower'
→ Fails when response.content is list instead of string

✅ V3 (FIXED):
──────────────────────────────────────────────────────────────────────────────
response = llm.invoke([message])

# First: Extract text safely
text = extract_text_from_response(response)  

# Then: Perform string operations
if "[no text found in image]" in text.lower():
    return (True, "[No text found in image]")
return (True, text)

WORKS: extract_text_from_response() handles all response types:
  - If string → return it directly
  - If list → join all text items
  - If dict → extract 'text' fields
  - Else → convert to string safely

══════════════════════════════════════════════════════════════════════════════
"""

print(comparison)


╔════════════════════════════════════════════════════════════════════════════╗
║                     CRITICAL FIX: Response Handling                        ║
╚════════════════════════════════════════════════════════════════════════════╝

❌ V2 (BROKEN):
──────────────────────────────────────────────────────────────────────────────
response = llm.invoke([message])
if '[no text found in image]' in response.content.lower():  # ← CRASH HERE
    return "[No text found in image]"
return response.content

ERROR: 'list' object has no attribute 'lower'
→ Fails when response.content is list instead of string

✅ V3 (FIXED):
──────────────────────────────────────────────────────────────────────────────
response = llm.invoke([message])

# First: Extract text safely
text = extract_text_from_response(response)  

# Then: Perform string operations
if "[no text found in image]" in text.lower():
    return (True, "[No text found in image]")
return (True, text)

WORKS: extract_text_from_response() handle

## Debug Mode - Detailed Failure Analysis

Enable this to see exactly why pages are failing

In [ ]:
def debug_page_in_detail(pdf_path: str, page_num: int, api_key: str):
    """
    Deep dive debug on a single page to show exactly what's happening.
    Shows: image extraction, encoding, API response, errors.
    """
    print(f"\n{'='*70}")
    print(f"DETAILED DEBUG - Page {page_num}")
    print(f"{'='*70}\n")
    
    if not os.path.exists(pdf_path):
        print(f"❌ PDF not found: {pdf_path}")
        return
    
    try:
        doc = fitz.open(pdf_path)
        if page_num < 1 or page_num > len(doc):
            print(f"❌ Page {page_num} out of range (1-{len(doc)})")
            doc.close()
            return
        
        page = doc.load_page(page_num - 1)
        print(f"✓ Loaded page {page_num}")
        
        # Step 1: Check for images on page
        image_list = page.get_images(full=True)
        print(f"\n[STEP 1] Image Detection:")
        print(f"  • Images found on page: {len(image_list)}")
        
        if not image_list:
            print(f"  ⚠️  NO IMAGES FOUND - This page might be text-only or blank")
            doc.close()
            return
        
        # Step 2: Try to extract each image
        print(f"\n[STEP 2] Image Extraction:")
        for img_idx, img_info in enumerate(image_list, 1):
            xref = img_info[0]
            print(f"\n  Image {img_idx}:")
            print(f"    - XRef ID: {xref}")
            
            try:
                base_image = page.parent.extract_image(xref)
                if not base_image:
                    print(f"    ❌ Extraction returned None")
                    continue
                
                image_bytes = base_image.get("image")
                if not image_bytes:
                    print(f"    ❌ No image data in extraction")
                    continue
                
                print(f"    ✓ Extracted {len(image_bytes):,} bytes")
                
                # Step 3: Check image properties
                print(f"\n  [STEP 3] Image Properties:")
                pil_image = Image.open(io.BytesIO(image_bytes))
                print(f"    - Size: {pil_image.size[0]}x{pil_image.size[1]} pixels")
                print(f"    - Mode: {pil_image.mode}")
                print(f"    - Format: {pil_image.format}")
                
                # Step 4: Convert to base64
                print(f"\n  [STEP 4] Base64 Encoding:")
                try:
                    img_base64 = image_to_base64(pil_image, format="JPEG")
                    print(f"    ✓ Encoded to {len(img_base64):,} characters base64")
                except Exception as e:
                    print(f"    ❌ Encoding failed: {e}")
                    continue
                
                # Step 5: Send to API
                print(f"\n  [STEP 5] Sending to Gemini API:")
                print(f"    - Model: {MODEL}")
                print(f"    - Temperature: {TEMPERATURE}")
                print(f"    - Rate limit delay: {RATE_LIMIT_DELAY}s")
                
                try:
                    llm = ChatGoogleGenerativeAI(
                        model=MODEL,
                        google_api_key=api_key,
                        temperature=TEMPERATURE
                    )
                    
                    message = HumanMessage(
                        content=[
                            {
                                "type": "text",
                                "text": "Extract all text from this image. If no text, respond: [No text found in image]",
                            },
                            {
                                "type": "image_url",
                                "image_url": f"data:image/jpeg;base64,{img_base64}",
                            },
                        ]
                    )
                    
                    print(f"    → Waiting {RATE_LIMIT_DELAY}s before API call...")
                    time.sleep(RATE_LIMIT_DELAY)
                    
                    print(f"    → Calling API...")
                    response = llm.invoke([message])
                    print(f"    ✓ API response received")
                    
                    # Step 6: Analyze response
                    print(f"\n  [STEP 6] Response Analysis:")
                    print(f"    - Response type: {type(response)}")
                    print(f"    - Has 'content': {hasattr(response, 'content')}")
                    
                    if hasattr(response, 'content'):
                        content = response.content
                        print(f"    - Content type: {type(content)}")
                        print(f"    - Content length: {len(str(content))}")
                        
                        if isinstance(content, str):
                            print(f"    - Content (String): {content[:200]}")
                        elif isinstance(content, list):
                            print(f"    - Content (List with {len(content)} items):")
                            for i, item in enumerate(content[:3]):  # Show first 3
                                print(f"      [{i}] {type(item)}: {str(item)[:100]}")
                        else:
                            print(f"    - Content (Other): {type(content)}")
                    
                    # Step 7: Extract text
                    print(f"\n  [STEP 7] Text Extraction:")
                    try:
                        extracted = extract_text_from_response(response)
                        print(f"    ✓ Text extracted: {len(extracted)} characters")
                        print(f"    - First 300 chars: {extracted[:300]}")
                        
                        if "[no text found" in extracted.lower():
                            print(f"    ⚠️  API reported no text found in image")
                        
                    except Exception as e:
                        print(f"    ❌ Extraction failed: {e}")
                
                except Exception as e:
                    print(f"    ❌ API call failed: {type(e).__name__}: {str(e)[:200]}")
            
            except Exception as e:
                print(f"    ❌ Unexpected error: {type(e).__name__}: {e}")
        
        doc.close()
        
    except Exception as e:
        print(f"❌ Fatal error: {type(e).__name__}: {e}")
    
    print(f"\n{'='*70}\n")


# Run debug on failing pages
print("\n" + "="*70)
print("DEBUGGING FAILING PAGES FROM YOUR RUN")
print("="*70)

failing_pages = [2, 4, 5]  # From your output above

for page_num in failing_pages:
    try:
        debug_page_in_detail(pdf_file, page_num, GOOGLE_API_KEY)
    except Exception as e:
        print(f"Debug failed for page {page_num}: {e}")
        import traceback
        traceback.print_exc()


DEBUGGING FAILING PAGES FROM YOUR RUN

DETAILED DEBUG - Page 2

✓ Loaded page 2

[STEP 1] Image Detection:
  • Images found on page: 1

[STEP 2] Image Extraction:

  Image 1:
    - XRef ID: 114
    ✓ Extracted 56,838 bytes

  [STEP 3] Image Properties:
    - Size: 1240x1624 pixels
    - Mode: RGB
    - Format: JPEG

  [STEP 4] Base64 Encoding:
    ✓ Encoded to 81,484 characters base64

  [STEP 5] Sending to Gemini API:
    - Model: gemini-3-flash-preview
    - Temperature: 0.3
    - Rate limit delay: 1.5s
    → Waiting 1.5s before API call...
    → Calling API...
    ❌ API call failed: ChatGoogleGenerativeAIError: Error calling model 'gemini-3-flash-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing detai



DETAILED DEBUG - Page 4

✓ Loaded page 4

[STEP 1] Image Detection:
  • Images found on page: 1

[STEP 2] Image Extraction:

  Image 1:
    - XRef ID: 120
    ✓ Extracted 66

In [ ]:
# Alternative: Run with verbose logging on a batch
def process_pdf_page_with_ocr_verbose(page: fitz.Page, llm_api_key: str) -> dict:
    """
    Process page with detailed logging of each step.
    Returns detailed dictionary with results and debug info.
    """
    page_num = page.number + 1
    results = {
        "page": page_num,
        "images_found": 0,
        "images_processed": 0,
        "images_successful": 0,
        "images_failed": 0,
        "failures": [],
        "text": "",
    }
    
    image_list = page.get_images(full=True)
    results["images_found"] = len(image_list)
    
    if not image_list:
        results["failures"].append("No images found on page")
        return results
    
    for img_index, img_info in enumerate(image_list, 1):
        xref = img_info[0]
        results["images_processed"] += 1
        
        try:
            # Extract
            base_image = page.parent.extract_image(xref)
            if not base_image:
                results["failures"].append(f"Image {img_index}: Extract returned None")
                results["images_failed"] += 1
                continue
            
            image_bytes = base_image.get("image")
            if not image_bytes:
                results["failures"].append(f"Image {img_index}: No image data")
                results["images_failed"] += 1
                continue
            
            # Encode
            pil_image = Image.open(io.BytesIO(image_bytes))
            img_base64 = image_to_base64(pil_image, format="JPEG")
            
            # OCR with detailed error catching
            success, ocr_text = get_ocr_text_from_image(img_base64, llm_api_key)
            
            if success:
                results["text"] += ocr_text + "\n\n"
                results["images_successful"] += 1
            else:
                results["failures"].append(f"Image {img_index}: {ocr_text}")
                results["images_failed"] += 1
                
        except Exception as e:
            results["failures"].append(f"Image {img_index}: {type(e).__name__}: {str(e)[:100]}")
            results["images_failed"] += 1
    
    return results


# Test verbose mode on first 5 pages
print("\n" + "="*70)
print("RUNNING FIRST 5 PAGES WITH VERBOSE LOGGING")
print("="*70 + "\n")

try:
    doc = fitz.open(pdf_file)
    
    for page_num in range(1, min(6, len(doc) + 1)):
        page = doc.load_page(page_num - 1)
        result = process_pdf_page_with_ocr_verbose(page, GOOGLE_API_KEY)
        
        print(f"Page {page_num}:")
        print(f"  • Images found: {result['images_found']}")
        print(f"  • Processed: {result['images_processed']}")
        print(f"  • Success: {result['images_successful']}")
        print(f"  • Failed: {result['images_failed']}")
        
        if result['failures']:
            print(f"  • Failure reasons:")
            for failure in result['failures']:
                print(f"    - {failure}")
        
        if result['text']:
            print(f"  • Text extracted: {len(result['text'])} characters")
        
        print()
    
    doc.close()
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

NameError: name 'fitz' is not defined

In [ ]:
# Quick diagnostic to check what's happening
print("\n" + "="*70)
print("QUICK DIAGNOSTIC - What's Causing Failures?")
print("="*70 + "\n")

diagnostic = {
    "empty_pages": [],
    "no_images": [],
    "extraction_errors": [],
    "api_errors": [],
    "no_text_in_image": [],
    "successful": [],
}

try:
    doc = fitz.open(pdf_file)
    
    # Check first 10 pages quickly
    for page_num in range(1, min(11, len(doc) + 1)):
        try:
            page = doc.load_page(page_num - 1)
            image_list = page.get_images(full=True)
            
            if not image_list:
                diagnostic["no_images"].append(page_num)
                continue
            
            # Try first image
            success = False
            for img_info in image_list:
                xref = img_info[0]
                try:
                    base_image = page.parent.extract_image(xref)
                    if not base_image:
                        diagnostic["extraction_errors"].append(page_num)
                        continue
                    
                    image_bytes = base_image.get("image")
                    if not image_bytes:
                        diagnostic["extraction_errors"].append(page_num)
                        continue
                    
                    # Try to OCR it
                    pil_image = Image.open(io.BytesIO(image_bytes))
                    img_base64 = image_to_base64(pil_image, format="JPEG")
                    
                    success_ocr, text_ocr = get_ocr_text_from_image(img_base64, GOOGLE_API_KEY)
                    
                    if success_ocr:
                        diagnostic["successful"].append(page_num)
                        success = True
                        break
                    else:
                        if "no text" in text_ocr.lower():
                            diagnostic["no_text_in_image"].append(page_num)
                        else:
                            diagnostic["api_errors"].append((page_num, text_ocr[:50]))
                        success = True
                        break
                except Exception as e:
                    diagnostic["api_errors"].append((page_num, str(e)[:50]))
                    success = True
                    break
            
        except Exception as e:
            diagnostic["extraction_errors"].append(page_num)
    
    doc.close()
    
    # Print results
    print("📊 DIAGNOSTIC RESULTS:\n")
    print(f"✅ Successful Pages: {diagnostic['successful']}")
    print(f"   → These pages extracted text correctly\n")
    
    print(f"⚠️  Pages with NO IMAGES: {diagnostic['no_images']}")
    print(f"   → Check if these are text-only pages or blank\n")
    
    print(f"❌ Extraction Errors: {diagnostic['extraction_errors']}")
    print(f"   → Image extraction from PDF failed\n")
    
    print(f"🔍 No Text in Image: {diagnostic['no_text_in_image']}")
    print(f"   → Images are present but contain no readable text\n")
    
    if diagnostic['api_errors']:
        print(f"⚡ API/OCR Errors: {len(diagnostic['api_errors'])} pages")
        for page_num, error in diagnostic['api_errors'][:3]:
            print(f"   • Page {page_num}: {error}")
        print()
    
    # Summary
    print("\n" + "="*70)
    print("SUMMARY:")
    print("="*70)
    print(f"Total pages checked: 10")
    print(f"Successful: {len(diagnostic['successful'])} ({len(diagnostic['successful'])*10}%)")
    print(f"No images found: {len(diagnostic['no_images'])}")
    print(f"Extraction errors: {len(diagnostic['extraction_errors'])}")
    print(f"No text detected: {len(diagnostic['no_text_in_image'])}")
    print(f"API errors: {len(diagnostic['api_errors'])}")
    
except Exception as e:
    print(f"Diagnostic failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
def pdf_to_ocr_knowledge_base_with_logging(pdf_path: str, output_path: str, start_pg: int, end_pg: int, api_key: str, log_file: str = None):
    """
    Process PDF with detailed logging to file and console.
    Logs every failure reason so you can see exactly what went wrong.
    
    Args:
        log_file: If provided, writes detailed logs to this file
    """
    # Validate inputs
    if not os.path.exists(pdf_path):
        print(f"ERROR: PDF file not found: {pdf_path}")
        return False
    
    # Setup logging
    log_lines = []
    
    def log_msg(msg, level="INFO"):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = f"[{timestamp}] [{level}] {msg}"
        log_lines.append(log_entry)
        print(log_entry)
    
    log_msg(f"Starting PDF processing with detailed logging")
    log_msg(f"PDF: {pdf_path}")
    log_msg(f"Output: {output_path}")
    
    # Open PDF
    try:
        doc = fitz.open(pdf_path)
        total_pages = len(doc)
        log_msg(f"PDF loaded: {total_pages} pages")
    except Exception as e:
        log_msg(f"Failed to open PDF: {e}", "ERROR")
        return False
    
    # Normalize page range
    actual_start = max(1, min(start_pg, total_pages))
    actual_end = min(max(start_pg, end_pg), total_pages)
    log_msg(f"Processing pages {actual_start}-{actual_end}")
    
    # Initialize tracking
    failure_log = {}  # page -> [reasons]
    stats = {
        "pages_processed": 0,
        "pages_successful": 0,
        "pages_with_content": 0,
        "pages_no_images": 0,
        "pages_extraction_error": 0,
        "pages_api_error": 0,
        "pages_no_text": 0,
        "images_successful": 0,
        "images_failed": 0,
        "total_characters": 0,
    }
    
    final_output = f"# Knowledge Base - {os.path.basename(pdf_path)}\n\n"
    final_output += f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    final_output += f"**Pages:** {actual_start}-{actual_end}\n"
    final_output += f"**Model:** {MODEL}\n\n---\n\n"
    
    # Process pages
    start_time = time.time()
    
    for page_num in range(actual_start, actual_end + 1):
        try:
            page = doc.load_page(page_num - 1)
            page_failures = []
            
            # Get images
            image_list = page.get_images(full=True)
            
            if not image_list:
                stats["pages_no_images"] += 1
                page_failures.append("No images found")
            else:
                log_msg(f"Page {page_num}: Found {len(image_list)} image(s)", "DEBUG")
                
                for img_idx, img_info in enumerate(image_list, 1):
                    xref = img_info[0]
                    
                    try:
                        # Extract
                        base_image = page.parent.extract_image(xref)
                        if not base_image:
                            stats["images_failed"] += 1
                            page_failures.append(f"Image {img_idx}: Extract=None")
                            continue
                        
                        image_bytes = base_image.get("image")
                        if not image_bytes:
                            stats["images_failed"] += 1
                            page_failures.append(f"Image {img_idx}: No data")
                            continue
                        
                        # Encode
                        pil_image = Image.open(io.BytesIO(image_bytes))
                        img_base64 = image_to_base64(pil_image, format="JPEG")
                        
                        # OCR
                        success, ocr_text = get_ocr_text_from_image(img_base64, api_key)
                        
                        if success:
                            final_output += f"## Page {page_num} - Image {img_idx}\n\n{ocr_text}\n\n"
                            stats["images_successful"] += 1
                            stats["total_characters"] += len(ocr_text)
                            stats["pages_with_content"] += 1
                        else:
                            stats["images_failed"] += 1
                            if "no text" in ocr_text.lower():
                                page_failures.append(f"Image {img_idx}: No text found")
                                stats["pages_no_text"] += 1
                            elif "rate limit" in ocr_text.lower():
                                page_failures.append(f"Image {img_idx}: Rate limited")
                                stats["pages_api_error"] += 1
                            else:
                                page_failures.append(f"Image {img_idx}: {ocr_text[:60]}")
                                stats["pages_api_error"] += 1
                    
                    except Exception as e:
                        stats["images_failed"] += 1
                        page_failures.append(f"Image {img_idx}: {type(e).__name__}: {str(e)[:50]}")
                        stats["pages_extraction_error"] += 1
            
            stats["pages_processed"] += 1
            if not page_failures:
                stats["pages_successful"] += 1
            
            failure_log[page_num] = page_failures
            
            # Progress
            elapsed = time.time() - start_time
            pages_done = page_num - actual_start + 1
            pages_total = actual_end - actual_start + 1
            rate = pages_done / elapsed if elapsed > 0 else 0
            eta_secs = (pages_total - pages_done) / rate if rate > 0 else 0
            
            success_count = stats["images_successful"]
            fail_count = stats["images_failed"]
            
            print(f"[{pages_done}/{pages_total}] Page {page_num}: {success_count}✓ {fail_count}✗ | ETA: {int(eta_secs//60)}m {int(eta_secs%60)}s")
            
            if page_failures:
                for failure in page_failures:
                    log_msg(f"  Page {page_num}: {failure}", "WARN")
        
        except Exception as e:
            log_msg(f"Page {page_num}: Unexpected error: {e}", "ERROR")
            stats["pages_processed"] += 1
            failure_log[page_num] = [str(e)]
    
    doc.close()
    
    # Add statistics to output
    final_output += f"\n---\n\n## Processing Statistics\n\n"
    final_output += f"- **Pages Processed:** {stats['pages_processed']}\n"
    final_output += f"- **Pages with Content:** {stats['pages_with_content']}\n"
    final_output += f"- **Pages Successful:** {stats['pages_successful']}\n"
    final_output += f"- **Pages with No Images:** {stats['pages_no_images']}\n"
    final_output += f"- **Pages with No Text:** {stats['pages_no_text']}\n"
    final_output += f"- **Pages with API Errors:** {stats['pages_api_error']}\n"
    final_output += f"- **Images Successfully Extracted:** {stats['images_successful']}\n"
    final_output += f"- **Images Failed:** {stats['images_failed']}\n"
    final_output += f"- **Total Characters:** {stats['total_characters']:,}\n"
    
    # Save output
    try:
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(final_output)
        log_msg(f"Output saved to: {output_path}")
    except Exception as e:
        log_msg(f"Failed to save output: {e}", "ERROR")
        return False
    
    # Save logs if requested
    if log_file:
        try:
            with open(log_file, "w", encoding="utf-8") as f:
                for line in log_lines:
                    f.write(line + "\n")
                
                # Add failure summary
                f.write("\n\n=== FAILURE SUMMARY ===\n\n")
                for page_num, failures in sorted(failure_log.items()):
                    if failures:
                        f.write(f"Page {page_num}:\n")
                        for failure in failures:
                            f.write(f"  - {failure}\n")
            
            log_msg(f"Detailed logs saved to: {log_file}")
        except Exception as e:
            log_msg(f"Failed to save logs: {e}", "ERROR")
    
    elapsed = time.time() - start_time
    log_msg(f"Processing completed in {int(elapsed//60)}m {int(elapsed%60)}s")
    
    return True


# Example: Run with detailed logging
print("\n" + "="*70)
print("RUNNING WITH DETAILED LOGGING (First 5 pages)")
print("="*70 + "\n")

log_output = "processing_debug.log"
pdf_to_ocr_knowledge_base_with_logging(
    pdf_file, 
    "output_ocr_debug.md", 
    1, 
    5,  # Just first 5 pages for testing
    GOOGLE_API_KEY,
    log_file=log_output
)

print(f"\n✓ Check '{log_output}' for detailed failure reasons")